<a href="https://colab.research.google.com/github/mancinigabriel/tcc-pece-assin2-llm-challenges/blob/main/notebooks/Bode_7B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/mancinigabriel/tcc-pece-assin2-llm-challenges.git

Cloning into 'tcc-pece-assin2-llm-challenges'...
remote: Enumerating objects: 210, done.
remote: Counting objects: 100% (210/210), done.
remote: Compressing objects: 100% (166/166), done.
remote: Total 210 (delta 117), reused 110 (delta 42), pack-reused 0 (from 0)
Receiving objects: 100% (210/210), 451.46 KiB | 8.06 MiB/s, done.
Resolving deltas: 100% (117/117), done.


#Preparando ambiente

In [2]:
import sys
import os

PROJECT_ROOT = os.path.abspath('/content/tcc-pece-assin2-llm-challenges')

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datetime import datetime, timezone, timedelta
from src import prompts
from src import utils
from src import data
from src import metrics
import pandas as pd
import random
import torch
import re
import time

In [5]:
cfg = utils.load_config(
    "/content/tcc-pece-assin2-llm-challenges/configs/base.yaml",
    "/content/tcc-pece-assin2-llm-challenges/configs/models/bode.yaml"
)

generation_args = cfg["generation"]

gpu = 'A100 RAM alta'

# Bode 7B

In [6]:
timing = {}

timing['inicio'] = utils.time_log()

model_id = 'recogna-nlp/bode-7b-alpaca-pt-br-no-peft'
model_name = "bode_7b"
quantizado = False
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

df_assin_2 = data.gera_df()

config.json:   0%|          | 0.00/672 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

pytorch_model-00002-of-00003.bin:   0%|          | 0.00/9.89G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/183 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [ ]:
timing['inicio_cons'] = utils.time_log()

df_assin_2_consistency_test = df_assin_2.head(500)

for j in range(5):
  for i in range(len(df_assin_2_consistency_test)):
    premissa = df_assin_2_consistency_test.iloc[i]['premise']
    hipotese = df_assin_2_consistency_test.iloc[i]['hypothesis']

    prompt = prompts.zero_shot_prompt(premissa, hipotese)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, **generation_args)
    prompt_len = inputs["input_ids"].shape[1]
    generated_tokens = output[0][prompt_len:]
    resp = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    column_name = f'test_{j}'

    df_assin_2_consistency_test.loc[i, column_name] = resp

    if i%100==0:
      print(f"{i} - {utils.time_log().strftime("%Y-%m-%d %H:%M:%S")}")

  df_assin_2_consistency_test[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2_consistency_test[column_name]))

df_assin_2_consistency_test.to_csv(f'/content/drive/MyDrive/Mestrado/TCC Pós/Dados/dados_{model_name}_consistencia.csv')

timing['fim_cons'] = utils.time_log()

/tmp/ipython-input-3760482454.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test.loc[i, column_name] = resp


0 - 2026-01-12 02:17:58
100 - 2026-01-12 02:18:05
200 - 2026-01-12 02:18:13
300 - 2026-01-12 02:18:20
400 - 2026-01-12 02:18:28


/tmp/ipython-input-3760482454.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2_consistency_test[column_name]))
/tmp/ipython-input-3760482454.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test.loc[i, column_name] = resp


0 - 2026-01-12 02:18:35
100 - 2026-01-12 02:18:42
200 - 2026-01-12 02:18:50
300 - 2026-01-12 02:18:57
400 - 2026-01-12 02:19:05


/tmp/ipython-input-3760482454.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2_consistency_test[column_name]))
/tmp/ipython-input-3760482454.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test.loc[i, column_name] = resp


...

/tmp/ipython-input-3760482454.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2_consistency_test[column_name]))
/tmp/ipython-input-3760482454.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test.loc[i, column_name] = resp


0 - 2026-01-12 02:19:49
100 - 2026-01-12 02:19:56
200 - 2026-01-12 02:20:03
300 - 2026-01-12 02:20:11
400 - 2026-01-12 02:20:18


/tmp/ipython-input-3760482454.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2_consistency_test[column_name]))
/tmp/ipython-input-3760482454.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test.loc[i, column_name] = resp


0 - 2026-01-12 02:20:25
100 - 2026-01-12 02:20:33
200 - 2026-01-12 02:20:40
300 - 2026-01-12 02:20:47
400 - 2026-01-12 02:20:55


/tmp/ipython-input-3760482454.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2_consistency_test[column_name]))


In [ ]:
timing['inicio_aplicacao_total'] = utils.time_log()

for i in range(len(df_assin_2)):
  premissa = df_assin_2.iloc[i]['premise']
  hipotese = df_assin_2.iloc[i]['hypothesis']

  prompt = prompts.zero_shot_prompt(premissa, hipotese)

  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
  output = model.generate(**inputs, **generation_args)
  prompt_len = inputs["input_ids"].shape[1]
  generated_tokens = output[0][prompt_len:]
  resp = tokenizer.decode(generated_tokens, skip_special_tokens=True)
  column_name = f'pred'

  df_assin_2.loc[i, column_name] = resp

  if i%100==0:
    print(f"{i} - {utils.time_log().strftime("%Y-%m-%d %H:%M:%S")}")

df_assin_2[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2[column_name]))
df_assin_2.to_csv(f'/content/drive/MyDrive/Mestrado/TCC Pós/Dados/dados_{model_name}.csv')

timing['fim'] = utils.time_log()

metrics_dict = {'acurácia': metrics.calculate_accuracy(df_assin_2),
           'consistência': metrics.compute_consistency(df_assin_2_consistency_test)}

log = utils.log(model_id, gpu, quantizado, generation_args, metrics_dict, timing)
utils.export_log(log, model_name)

0 - 2026-01-12 02:21:04
100 - 2026-01-12 02:21:11
200 - 2026-01-12 02:21:18
300 - 2026-01-12 02:21:26
400 - 2026-01-12 02:21:33
500 - 2026-01-12 02:21:40
600 - 2026-01-12 02:21:48
700 - 2026-01-12 02:21:55
800 - 2026-01-12 02:22:02
900 - 2026-01-12 02:22:09
1000 - 2026-01-12 02:22:17
1100 - 2026-01-12 02:22:24
1200 - 2026-01-12 02:22:31
1300 - 2026-01-12 02:22:39
1400 - 2026-01-12 02:22:46
1500 - 2026-01-12 02:22:53
1600 - 2026-01-12 02:23:01
1700 - 2026-01-12 02:23:08
1800 - 2026-01-12 02:23:15
1900 - 2026-01-12 02:23:23
2000 - 2026-01-12 02:23:30
2100 - 2026-01-12 02:23:37
2200 - 2026-01-12 02:23:45
2300 - 2026-01-12 02:23:52
2400 - 2026-01-12 02:23:59
2500 - 2026-01-12 02:24:06
2600 - 2026-01-12 02:24:14
2700 - 2026-01-12 02:24:21
2800 - 2026-01-12 02:24:28
2900 - 2026-01-12 02:24:36
3000 - 2026-01-12 02:24:43
3100 - 2026-01-12 02:24:50
3200 - 2026-01-12 02:24:58
3300 - 2026-01-12 02:25:05
3400 - 2026-01-12 02:25:13
3500 - 2026-01-12 02:25:20
3600 - 2026-01-12 02:25:28
3700 - 2026-0

'Arquivo salvo em /content/drive/MyDrive/Mestrado/TCC Pós/Dados/logs/log_bode_7b_20260112_023227.json às 2026-01-12 02:32:27'

In [8]:
timing = {}

timing['inicio'] = utils.time_log()
timing['inicio_cons'] = utils.time_log()
timing['inicio_aplicacao_total'] = utils.time_log()

df_sintetico = data.gera_df_sintetico()

for i in range(len(df_sintetico)):
  premissa = df_sintetico.iloc[i]['premissa']
  hipotese = df_sintetico.iloc[i]['hipotese']

  prompt = prompts.zero_shot_prompt(premissa, hipotese)

  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
  output = model.generate(**inputs, **generation_args)
  prompt_len = inputs["input_ids"].shape[1]
  generated_tokens = output[0][prompt_len:]
  resp = tokenizer.decode(generated_tokens, skip_special_tokens=True)
  column_name = f'pred'

  df_sintetico.loc[i, column_name] = resp

df_sintetico[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_sintetico[column_name]))

timing['fim_cons'] = utils.time_log()
timing['fim'] = utils.time_log()

metrics_dict = {'acurácia_df_sintetico': metrics.calculate_accuracy(df_sintetico)}

log = utils.log(model_id, gpu, quantizado, generation_args, metrics_dict, timing)
utils.export_log(log, f"{model_name}_df_sintetico")
df_sintetico.to_csv(f'/content/drive/MyDrive/Mestrado/TCC Pós/Dados/dados_{model_name}_df_sintetico.csv')